# Predykcja przeżycia pasażerów Titanica

Celem notatnika jest uporządkowane przeprowadzenie eksploracji danych, przygotowanie cech, wytrenowanie modelu regresji logistycznej oraz zapisanie końcowego artefaktu `model.pkl`, który będzie wykorzystywany przez aplikację FastAPI.

### 1. Import bibliotek

W tym miejscu importujemy biblioteki potrzebne do analizy danych, wizualizacji, przygotowania zbioru oraz treningu i oceny modelu.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")

### 2. Przygotowanie danych do analizy

Wykorzystamy zbiór danych **Titanic - Machine Learning from Disaster**. Zmienna `Survived` informuje, czy pasażer przeżył katastrofę. Do przewidywania tej wartości wykorzystamy model regresji logistycznej.

Ścieżki są wyznaczane względem katalogu projektu, dzięki czemu notatnik można uruchomić zarówno z katalogu głównego, jak i z katalogu `research`.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "research":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "DSP_6.csv"
MODEL_PATH = PROJECT_ROOT / "artifacts" / "model.pkl"

train = pd.read_csv(DATA_PATH)
train.head()

Najważniejsze kolumny zbioru:

- `Survived` - informacja o przeżyciu: 0 oznacza brak przeżycia, 1 oznacza przeżycie,
- `Pclass` - klasa pasażera,
- `Age` - wiek pasażera,
- `SibSp` - liczba rodzeństwa lub małżonków na pokładzie,
- `Parch` - liczba rodziców lub dzieci na pokładzie,
- `Fare` - cena biletu,
- `Sex` - płeć pasażera,
- `Embarked` - port zaokrętowania: C, Q lub S.

In [ ]:
print(f"Liczba obserwacji: {train.shape[0]}")
print(f"Liczba kolumn: {train.shape[1]}")
train.info()

In [ ]:
train.describe(include="all").T

#### 2.1. Brakujące dane

Na początku sprawdzimy, w których kolumnach znajdują się brakujące wartości. Mapa ciepła pozwala szybko zauważyć ich rozmieszczenie.

In [ ]:
plt.figure(figsize=(12, 5))
sns.heatmap(train.isnull(), yticklabels=False, cbar=False, cmap="viridis")
plt.title("Brakujące wartości w zbiorze Titanic")
plt.show()

In [ ]:
missing_values = pd.DataFrame({
    "liczba_braków": train.isna().sum(),
    "procent_braków": (train.isna().sum() / len(train) * 100).round(2),
})
missing_values[missing_values["liczba_braków"] > 0].sort_values(
    "procent_braków", ascending=False
)

Kolumna `Cabin` zawiera zbyt wiele braków i nie będzie wykorzystywana przez model. Brakujące wartości `Age` uzupełnimy medianą, a brakujące wartości `Embarked` najczęściej występującym portem. Pozostałe kolumny tekstowe, takie jak imię i numer biletu, również nie będą cechami modelu.

#### 2.2. Eksploracja danych

Przed przygotowaniem modelu sprawdzimy podstawowe zależności między cechami pasażerów a informacją o przeżyciu.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=train, x="Survived", ax=axes[0])
axes[0].set_title("Liczba osób według przeżycia")

sns.countplot(data=train, x="Survived", hue="Sex", ax=axes[1])
axes[1].set_title("Przeżycie według płci")

sns.countplot(data=train, x="Survived", hue="Pclass", ax=axes[2])
axes[2].set_title("Przeżycie według klasy pasażera")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=train, x="Age", hue="Survived", kde=True, bins=25, ax=axes[0])
axes[0].set_title("Rozkład wieku według przeżycia")

sns.boxplot(data=train, x="Pclass", y="Fare", hue="Survived", ax=axes[1])
axes[1].set_title("Cena biletu według klasy i przeżycia")
axes[1].set_ylim(0, 200)

plt.tight_layout()
plt.show()

In [ ]:
survival_by_group = train.groupby(["Sex", "Pclass"])["Survived"].agg(
    liczba_pasażerów="count",
    współczynnik_przeżycia="mean",
)
survival_by_group["współczynnik_przeżycia"] = survival_by_group[
    "współczynnik_przeżycia"
].round(3)
survival_by_group

Eksploracja pokazuje, że płeć oraz klasa pasażera są silnie związane z przeżyciem. Wiek, liczba członków rodziny i cena biletu również mogą dostarczać modelowi przydatnych informacji.

#### 2.3. Czyszczenie danych i konwersja zmiennych

Regresja logistyczna oczekuje danych liczbowych i nie przyjmuje brakujących wartości. Wybieramy więc potrzebne kolumny, uzupełniamy braki oraz zamieniamy zmienne `Sex` i `Embarked` na zmienne binarne.

In [ ]:
model_data = train[
    ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]
].copy()

age_median = float(model_data["Age"].median())
fare_median = float(model_data["Fare"].median())
embarked_mode = str(model_data["Embarked"].mode().iloc[0])

model_data["Age"] = model_data["Age"].fillna(age_median)
model_data["Fare"] = model_data["Fare"].fillna(fare_median)
model_data["Embarked"] = model_data["Embarked"].fillna(embarked_mode)

print(f"Mediana wieku: {age_median}")
print(f"Mediana ceny biletu: {fare_median}")
print(f"Najczęstszy port zaokrętowania: {embarked_mode}")

In [ ]:
model_data = pd.get_dummies(
    model_data,
    columns=["Sex", "Embarked"],
    dtype="int",
    drop_first=True,
)

model_data.head()

In [ ]:
FEATURE_COLUMNS = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Sex_male",
    "Embarked_Q",
    "Embarked_S",
]

X = model_data[FEATURE_COLUMNS]
y = model_data["Survived"].astype(int)

assert not X.isna().any().any(), "Dane modelu nie mogą zawierać braków."
print(f"Rozmiar macierzy cech: {X.shape}")
X.head()

### 3. Trening modelu regresji logistycznej

Dzielimy dane na część treningową i testową. Parametr `stratify=y` zachowuje podobny udział osób, które przeżyły, w obu częściach zbioru.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f"Dane treningowe: {X_train.shape}")
print(f"Dane testowe: {X_test.shape}")

In [ ]:
logmodel = LogisticRegression(max_iter=500)
logmodel.fit(X_train, y_train)

predictions = logmodel.predict(X_test)
probabilities = logmodel.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")

#### 3.1. Ocena modelu

Raport klasyfikacji przedstawia precision, recall i F1-score. Macierz pomyłek pokazuje, ile predykcji model przypisał poprawnie i błędnie do każdej klasy.

In [ ]:
print(classification_report(y_test, predictions))
confusion_matrix(y_test, predictions)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap="Blues")
plt.title("Macierz pomyłek dla regresji logistycznej")
plt.show()

#### 3.2. Interpretacja współczynników

Dodatni współczynnik zwiększa przewidywane prawdopodobieństwo przeżycia, natomiast ujemny je zmniejsza. Wartości współczynników mają różne skale, dlatego traktujemy je przede wszystkim jako informację o kierunku wpływu cechy.

In [ ]:
coefficients = pd.DataFrame({
    "cecha": logmodel.feature_names_in_,
    "współczynnik": logmodel.coef_[0],
}).sort_values("współczynnik", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=coefficients, x="współczynnik", y="cecha")
plt.axvline(0, color="black", linewidth=1)
plt.title("Współczynniki modelu regresji logistycznej")
plt.show()

coefficients

### 4. Trening modelu końcowego i zapis artefaktu

Po ocenie modelu trenujemy końcową wersję na całym przygotowanym zbiorze. Dzięki temu artefakt wykorzystywany przez aplikację korzysta ze wszystkich dostępnych obserwacji. Model zapisujemy za pomocą biblioteki `pickle`.

In [ ]:
final_model = LogisticRegression(max_iter=500)
final_model.fit(X, y)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(final_model, model_file)

print(f"Model został zapisany w: {MODEL_PATH}")
print(f"Cechy modelu: {list(final_model.feature_names_in_)}")

### 5. Weryfikacja zapisanego modelu

Na koniec ponownie wczytujemy artefakt i wykonujemy przykładową predykcję. Potwierdza to, że zapisany plik może zostać poprawnie wykorzystany przez backend.

In [ ]:
with MODEL_PATH.open("rb") as model_file:
    loaded_model = pickle.load(model_file)

example_passenger = pd.DataFrame(
    [[3, 22.0, 1, 0, 7.25, 1, 0, 1]],
    columns=FEATURE_COLUMNS,
)

example_prediction = int(loaded_model.predict(example_passenger)[0])
example_probability = float(loaded_model.predict_proba(example_passenger)[0, 1])

print(f"Predykcja przeżycia: {example_prediction}")
print(f"Prawdopodobieństwo przeżycia: {example_probability:.2%}")